In [ ]:
#importing packages and vars

from pyspark.sql.session import SparkSession
import pyspark.sql.functions as f

import os
from dotenv import load_dotenv

#load environment variables
env = load_dotenv(dotenv_path='/mnt/d/linux/project/vars/.env')

#env vars for postgresql connection
mssqlsource = os.getenv('MSSQLSOURCE')
mssqluser = os.getenv('MSSQL_SPARK_USER')
mssqlpwd = os.getenv('MSSQL_SPARK_PWD')
mssqldriver = os.getenv('MSSQL_DRIVER')
sqlserverjar = os.getenv('SQLSERVER_JAR')
mssql_dwdest = os.getenv('MSSQL_CONN')

In [ ]:
spark = ((SparkSession
            .builder
            .appName('dimcountry')
            .config('spark.jar', f'{sqlserverjar}')
            .config('spark.driver.extraClassPath', f'{sqlserverjar}')
            .config('spark.executor.extraClassPath', f'{sqlserverjar}')
            .getOrCreate()
            
        ))

In [ ]:
#load customer data from source table

tmp_renamedrop_df = (( spark
                        .read
                        .format('jdbc')
                        .option('url', mssqlsource)
                        .option('driver', mssqldriver)
                        .option('user', mssqluser)
                        .option('password', mssqlpwd)
                        .option('query', 
                                """ 
                                
                                    select

                                        customerid,
                                        customercategoryid,
                                        buyinggroupid,
                                        billtocustomerid,
                                        split_part(customername, '(', 1)as firstname,
		                                replace(split_part(customername, '(', 2), ')', '') as lastnameOrBranch,
                                        replace(split_part(customername, ',', 2),')', '') as place,
                                        accountopeneddate,
                                        creditlimit,
                                        standarddiscountpercentage,
                                        paymentdays,
                                        primarycontactpersonid,
                                        alternatecontactpersonid,
                                        deliverymethodid,
                                        deliverycityid,
                                        postalcityid,
                                        deliverypostalcode,
                                        deliveryaddressline1,
                                        deliveryaddressline2,
                                        postaladdressline1,
                                        postaladdressline2,
                                        isstatementsent,
                                        isoncredithold
                                        
                                    from sales.customers
                                
                                """)
                        
                    )).load()

In [ ]:
#load customercategory data from source table
tmp_custmercategory = (( spark
                            .read
                            .format('jdbc')
                            .option('url', mssqlsource)
                            .option('driver', mssqldriver)
                            .option('user', mssqluser)
                            .option('password', mssqlpwd)
                            .option('query', 
                                    
                                    """ 
                                        select

                                            customercategoryid,
                                            customercategoryname
                                        
                                        from sales.customercategories
                                        
                                    """)
                        )).load()

In [ ]:
tmp_buyinggroups = (( spark
                        .read
                        .format('jdbc')
                        .option('url', mssqluser)
                        .option('driver', mssqldriver)
                        .option('user', mssqluser)
                        .option('password', mssqlpwd)
                        .option('query', 
                                
                                """
                                    select
                                    
                                        buyinggroupid,
                                        buyinggroupname
                                        
                                    from sales.buyinggroups
                                        
                                """)
                        )).load()

In [ ]:
#joining dataframes

joined_frames = (( tmp_renamedrop_df
                    .join(tmp_custmercategory, 'customercategoryid', 'left')
                    .join(tmp_buyinggroups, 'buyinggroupid', 'left')
                ))

In [ ]:
rename_df = ((
                joined_frames
                    .select('*', 
                            f.trim('firstname').alias('firstname_n'),
                            f.trim('lastnameOrbranch').alias('lastname'),
                            f.coalesce('creditlimit', f.lit(0.00)).alias('creditlimit_n'),
                            f.coalesce('buyinggroupid', f.lit(0)).alias('buyergroup_id'),
                            f.coalesce('alternatecontactpersonid', f.lit(0)).alias('alternatecontact_id')
                        )
))

In [ ]:
renamedrop_df = ((rename_df
                     .drop('creditlimit')
                     .drop('buyinggroupid')
                     .drop('alternatecontactpersonid')
                     .drop('firstname')
                     .drop('lastnameOrBranch')
                     
                     #rename columns
                     .withColumnRenamed('creditlimit_n', 'creditlimit')
                     .withColumnRenamed('buyergroup_id', 'buyergroupid')
                     .withColumnRenamed('alternatecontact_id', 'alternatecontactid')
                     .withColumnRenamed('standarddiscountpercentage', 'discountpercentage')
                     .withColumnRenamed('primarycontactpersonid', 'primarycontactid')
                     .withColumnRenamed('deliverypostalcode', 'postalcode')
                     .withColumnRenamed('deliveryaddressline1', 'primaryaddress')
                     .withColumnRenamed('deliveryaddressline2', 'alternateaddress')
                     .withColumnRenamed('firstname_n', 'firstname')
                     .withColumnRenamed('postaladdressline1', 'postaladdress')
                     .withColumnRenamed('postaladdressline2', 'alternatepostaladdress')
                     
))

In [ ]:
df = renamedrop_df.withColumn('lastname', f.split('lastname', '[^A-Za-z0-9 '']'))

renamedrop_df = df.withColumn('lastname', df['lastname'].getItem(0))


In [ ]:
#organizing columns

customerframe= ((
                renamedrop_df
                    .select('customerid',
                            'customercategoryid',
                            'billtocustomerid',
                            'buyergroupid',
                            'firstname',
                            'lastname',
                            'customercategoryname',
                            'buyinggroupname',
                            'accountopeneddate',
                            'paymentdays',
                            'creditlimit',
                            'discountpercentage',
                            'primarycontactid',
                            'alternatecontactid',
                            'deliverymethodid',
                            'deliverycityid',
                            'postalcityid',
                            'postalcode',
                            'primaryaddress',
                            'alternateaddress',
                            'postaladdress',
                            'alternatepostaladdress',
                            'place',
                            'isoncredithold',
                            'isstatementsent'
                            ).where('firstname not like "Tailspin%" and firstname not like "Wingtip%" ')
                    

))

In [ ]:
#change boolean for 'binary' and empty for nulls
customer = ((
    customerframe
        .withColumn('isoncredithold', f.when(renamedrop_df['isoncredithold'] == 'true', 1).otherwise(0))
        .withColumn('isstatementsent', f.when(renamedrop_df['isstatementsent'] == 'true', 1).otherwise(0))
        .withColumn('buyinggroupname', f.expr('coalesce(buyinggroupname, "Personal Customer")'))
        .withColumn('place', f.when(renamedrop_df['place'] == "", None).otherwise(renamedrop_df['place']))
        
        #normalize null values without rename column
        .withColumn('place', f.expr('coalesce(place, "Not Available")'))
        .withColumn('elt_buyergroup_hashid', f.expr('coalesce(elt_buyergroup_hashid, "Not Available")'))
        .withColumn('buyergroup_hashid', f.expr('coalesce(buyergroup_hashid, "Not Available")'))
        .withColumnRenamed('firstname', 'customername')
        .drop('lastname')
)).show()

In [ ]:
#create store table
storeframe = (( 
               renamedrop_df
                .select(    'customerid',
                            'customercategoryid',
                            'billtocustomerid',
                            'buyergroupid',
                            'firstname',
                            'lastname',
                            'customercategoryname',
                            'buyinggroupname',
                            'accountopeneddate',
                            'paymentdays',
                            'creditlimit',
                            'discountpercentage',
                            'primarycontactid',
                            'alternatecontactid',
                            'deliverymethodid',
                            'deliverycityid',
                            'postalcityid',
                            'postalcode',
                            'primaryaddress',
                            'alternateaddress',
                            'postaladdress',
                            'alternatepostaladdress',
                            'place',
                            'isoncredithold',
                            'isstatementsent')
                .where('firstname like "Tailspin%" or firstname like "Wingtip%"')
                .withColumnRenamed('lastname', 'businessunity')
               ))

In [ ]:
#write to table

((
    customer
        .write
        .format('jdbc')
            .option('url', mssql_dwdest)
            .option('driver', mssqldriver)
            .option('user', mssqluser)
            .option('password', mssqlpwd)
            .option('dbtable', 'sales.testeinsert')
        .save(mode= 'overwrite')
))

In [ ]:
#write to table

((
    tmp_custmercategory
        .write
        .format('jdbc')
            .option('url', mssql_dwdest)
            .option('driver', mssqldriver)
            .option('user', mssqluser)
            .option('password', mssqlpwd)
            .option('dbtable', 'sales.testeinsert')
        .save(mode= 'overwrite')
))

In [ ]:
#write to table

((
    storeframe
        .write
        .format('jdbc')
            .option('url', mssql_dwdest)
            .option('driver', mssqldriver)
            .option('user', mssqluser)
            .option('password', mssqlpwd)
            .option('dbtable', 'sales.testeinsert')
        .save(mode= 'overwrite')
))